**Edited 2026-08-22:**
- Fixed `death_day` to be recomputed via a fresh tz-consistent DuckDB `DATE_DIFF` query, so a midnight-truncated same-day death no longer wrongly drops the patient from the whole 28-day at-risk series.
- Fixed the DNI exclusion to require `dni.start_dttm >= admission_dttm` of the current encounter_block, so an unrelated prior encounter's DNI status can no longer exclude a later encounter.
- Fixed the Step 4g `daily`/`provider_daily` merge to join on formatted date strings instead of `datetime64` columns, avoiding a tz-dtype mismatch.
- Fixed the LPV first-24h indicator to restrict its denominator to patients actually at risk on Day 1.
- Added Step 7 (Evidence-Based Practice Indicators): proning, LPV first 24h, LPV subsequent days, hypoglycemia avoidance.
- Replaced the proning indicator's P/F+PEEP proxy with `ards_classifier.R` (PROSEVA-based ARDS classifier), invoked via subprocess as in `ltvv_wrangler.ipynb`.

# Questions

- at-risk: death currently ineligible (no longer at risk)
- not cliffable: clif_admissions

# Plan

## Objective

Estimate, for each day of a 28-day window (Day 1 → Day 28), the number of eligible encounters and VFD-28 (ventilator-free days at 28 days) summary statistics among mechanically ventilated patients — stratified by ICU type and by the year/ICU-type-specific pool of eligible providers — to support power calculations for a study of provider-level practice variation in mechanical ventilation management.

**Reference:** Yehya N, et al. *Reappraisal of Ventilator-Free Days in Critical Care Research.* Am J Respir Crit Care Med. 2019. https://pmc.ncbi.nlm.nih.gov/articles/PMC6812447/

## Cohort Eligibility

Unit of analysis: a **stitched encounter** (`encounter_block`, clifpy `stitch_encounters`, 6h window; Step 0), not a raw `hospitalization_id` — two hospitalizations of the same patient within 6h (e.g., ED-to-inpatient transfer, quick readmission) are treated as one continuous encounter.

**Inclusion**
- Adults, `age_at_admission` ≥ 18 (at the encounter's first hospitalization)
- Invasive mechanical ventilation (`device_category == IMV`) while physically in an ICU (`location_category == icu`) — MV started outside the ICU doesn't count until it continues into an ICU-located record
- First ICU MV episode of the encounter only; Day 1 = the start of that episode

**Exclusion**
- ECMO at MV onset (not ECMO at any later point), evaluated at MV onset only, never looking forward
- Tracheostomy already in place at MV onset — operationalized as a trach documented within 24h of `mv_start_dttm`
- Cardiac arrest or anoxic brain injury present on admission — POA-only ICD-10 `I46.x` / `G93.1` from `hospital_diagnosis`
- DNI status before MV start — latest `code_status` between the encounter's own admission and MV start, for DNR/DNI, DNAR/DNI, or DNI_only (plain DNR/DNAR/UDNR are not excluded — CLIF's DNR means "no CPR but do intubate"). Bounded to the current encounter so a DNI status from an unrelated prior encounter can't exclude this one.

**Note:** these exclusion definitions (POA-based cardiac arrest/anoxic injury, DNI-before-MV) are a first pass and may not exactly match the grant protocol's exclusion language (e.g., anoxic injury documented within 48h of ventilation vs. present on admission; comfort-measures-only transition vs. DNI before MV) — pending confirmation with the study team.

## VFD-28 Definition (Yehya et al. 2019, with one deviation)

Computed once per encounter, over a fixed 28-day window from Day 1 (MV initiation):

- VFD-28 = 0 if the patient dies within 28 days, is still on IMV at Day 28, or no final extubation is observed before censoring
- VFD-28 = 28 − x if liberated on day x (extubated and off positive-pressure support, no subsequent reintubation)
- Reintubation: counted from the *final* extubation, not the first
- Deaths after Day 28 are censored (ignored)
- **Deviation:** no minimum sustained-liberation duration is required. Yehya et al. recommend >48h off support before crediting a "successful extubation," but that threshold targets avoiding credit for reintubated patients, not patients discharged shortly after a clean extubation — so any final, non-reintubated extubation counts immediately, however briefly observed before censoring.
- **Discharge censoring (simplifying assumption):** no ventilation status is tracked past discharge; VFD-28 is locked in as of the discharge date.

VFD-28 is fixed per encounter — it is not recomputed for each day of the at-risk loop; only the *set* of at-risk encounters changes as the loop advances.

**Summary statistics per stratum:** mean, SD, median, IQR, proportion at 0, proportion at 28.

## Provider Eligibility & ICU-type Stratification

A provider is **eligible** for a given ICU type/year if (a) they've initiated ≥ `ELIGIBLE_PROVIDER_MIN_INITIATIONS` ICU IMV episodes globally (a volume floor; Step 3a), and (b) they were active in that `icu_type` during that year. Eligibility is nested by unit, so counts are stratified by (ICU type, year). No role/title field exists in the real provider data, so "eligible" means volume-qualified, not specifically attending-level.

**Two eligible-provider counts, both reported:**
- **Per patient, per calendar day:** distinct eligible providers who had ≥1 patient attributed to them, in that patient's hospital/ICU stratum, on the *calendar date* their VFD day fell on (not a relative VFD day — two patients on their own "Day 5" almost never share a calendar date). Attached per at-risk patient-day, then reported as the median/IQR of that count, by (hospital, icu_stratum, day) — the actual daily staffing pool a patient could have drawn from.
- **Per ICU/year (roster):** eligible-provider list and count per (hospital, icu_type, year) — the annual maximum pool, not the daily one.

**Stratification scope:**
- East Bank (academic): by ICU type *and* year — medical, surgical, neuro, cardiovascular
- Community hospitals: by hospital and year only (not split by unit)

Patient and provider counts are reported at the same stratification granularity used in the eventual trial-level analysis.

## Daily Landmark Workflow

For each Day *d* = 1…28 and each stratum (academic: hospital × ICU type × year; community: hospital × year), an encounter is **at risk** if, at the start of Day *d*, it is:
- Alive, not yet discharged, still on the ventilator (real-time device state), still in the unit, and covered by an eligible provider
- A member of the base cohort

For each day/stratum, report: N at risk, N eligible providers per patient (median, IQR) and per ICU/year (roster count), and VFD-28 summary statistics for the at-risk set.

The at-risk set shrinks day to day as encounters drop out (liberation, death, discharge, transfer), but each encounter's VFD-28 value is fixed and keeps contributing to every day's stats through its last at-risk day. "Still on the ventilator" is part of at-risk — not just death/discharge — because the daily count tracks exposure to an active provider ventilator-management decision, not the outcome itself.

## Deliverables

- **Cohort table** — the final analytic cohort after all exclusions (age, ECMO/trach-at-onset, cardiac arrest/anoxic injury POA, DNI-before-MV)
- **Roster table** — eligible providers by (year, ICU type) for academic sites, (year, hospital) for community sites
- **Daily table** — N at risk, N eligible providers per patient (median, IQR) and per ICU/year (roster count), and VFD-28 summary statistics, by (day 1–28) × stratum
- **QC summary** — cohort/exclusion counts and diagnostic statistics from every pipeline step, in one file
- Patient-level data (cohort table) stays local; aggregate tables are shareable

# Import Libraries

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import numpy as np
import pandas as pd
import duckdb
import clifpy
from clifpy.utils.stitching_encounters import stitch_encounters

# Global Settings

In [ ]:
pd.set_option('display.max_columns', None)

os.makedirs('output_no_share', exist_ok=True)
os.makedirs('output_to_box', exist_ok=True)

con = duckdb.connect(database='output_no_share/vfd28_pipeline.duckdb')

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["data_directory"]
file_type = cfg["filetype"]
time_zone = cfg["timezone"]
site_name = cfg.get("site_name", "site")

print(f"CLIF filepath: {clif_path}")
print(f"Filetype: {file_type}")
print(f"Timezone: {time_zone}")
print(f"Site name: {site_name}")

con.execute(f"SET TimeZone = '{time_zone}'")

# File paths

In [ ]:
ext = file_type   # "parquet" or "csv"
hospitalization_path = f"{clif_path}/clif_hospitalization.{ext}"
adt_path              = f"{clif_path}/clif_adt.{ext}"
patient_path          = f"{clif_path}/clif_patient.{ext}"
vent_path             = f"{clif_path}/clif_respiratory_support.{ext}"
ecmo_path             = f"{clif_path}/clif_ecmo_mcs.{ext}"
hosp_diagnosis_path   = f"{clif_path}/clif_hospital_diagnosis.{ext}"
dnr_path              = f"{clif_path}/clif_code_status.{ext}"
provider_path         = f"{clif_path}/clif_provider.{ext}"
labs_path             = f"{clif_path}/clif_labs.{ext}"
vitals_path           = f"{clif_path}/clif_vitals.{ext}"
position_path         = f"{clif_path}/clif_position.{ext}"


In [ ]:
REQUIRED = {
    hospitalization_path: ["hospitalization_id", "patient_id", "admission_dttm", "discharge_dttm",
                            "age_at_admission", "admission_type_category", "discharge_category"],
    adt_path:             ["hospitalization_id", "hospital_id", "in_dttm", "out_dttm",
                            "location_category", "hospital_type", "location_type"],
    vent_path:            ["hospitalization_id", "recorded_dttm", "device_category", "tracheostomy",
                            "tidal_volume_set", "tidal_volume_obs"],
    patient_path:         ["patient_id", "death_dttm", "sex_category"],
    ecmo_path:            ["hospitalization_id", "recorded_dttm", "device_category"],
    hosp_diagnosis_path:  ["hospitalization_id", "diagnosis_code", "poa_present", "diagnosis_primary"],
    dnr_path:             ["patient_id", "start_dttm", "code_status_category"],
    provider_path:        ["hospitalization_id", "recorded_date", "recorded_hour", "prov_npi"],
    labs_path:            ["hospitalization_id", "lab_category", "lab_value_numeric", "lab_collect_dttm"],
    vitals_path:          ["hospitalization_id", "vital_category", "vital_value", "recorded_dttm"],
    position_path:        ["hospitalization_id", "recorded_dttm", "position_category"],
}

errors = []
for path, required_cols in REQUIRED.items():
    name = os.path.basename(path)
    if not os.path.exists(path):
        errors.append(f"  MISSING FILE:  {name}")
        continue
    actual_cols = set(duckdb.sql(f"SELECT * FROM '{path}' LIMIT 0").columns)
    missing = [c for c in required_cols if c not in actual_cols]
    if missing:
        errors.append(f"  MISSING COLS:  {name} -> {missing}")
    else:
        print(f"  \u2713 {name}")

if errors:
    print("\nPRE-FLIGHT FAILED:")
    for e in errors:
        print(e)
    raise RuntimeError("Fix the above issues before running the notebook.")
else:
    print("\nAll required CLIF tables present and columns verified. Ready to run.")

# Pipeline Constants

All tunable rules from the Plan are centralized here — change once, applies everywhere below.

In [ ]:
STITCH_TIME_INTERVAL_HOURS = 6   # hours between discharge and next admission to link two
                                  # hospitalizations into one encounter_block (Step 0)

ECMO_ONSET_BUFFER_HOURS  = 0     # ECMO at/before MV start (+buffer) = "on ECMO at onset" (excluded).
                                  # Kept at 0 so ECMO initiated shortly AFTER MV start does not
                                  # exclude the encounter (only ECMO-at-onset is excluded).
TRACH_ONSET_WINDOW_HOURS = 24    # trach documented within this window of MV start = "in place at onset"
VFD_WINDOW_DAYS          = 28

# A provider must have initiated at least this many ICU IMV episodes, globally across the whole
# dataset, to count as an "eligible provider" anywhere downstream (Step 3 roster and Step 4's
# per-patient coverage check). "Initiated" = the provider covering the patient at the exact moment
# of their first ICU IMV record (see Step 3a).
ELIGIBLE_PROVIDER_MIN_INITIATIONS = 25

# device_category values representing OFF positive-pressure support (liberated), regardless of
# tracheostomy status. Stored lower-case to match the case-insensitive comparisons used throughout.
OFF_SUPPORT_DEVICES = ("room air", "nasal cannula", "face mask", "trach collar")

# Noninvasive support devices — per Yehya et al. 2019, NIV/HFNC should not be held against
# liberation for a standard (non-tracheostomized) patient; the stricter "off ALL positive-pressure
# support" standard applies only to tracheostomized patients. So NIV_DEVICES count as OFF unless
# tracheostomy=1 at that row. IMV is always ON; any unrecognized device_category defaults to ON.
NIV_DEVICES = ("nippv", "high flow nc")

# --- Step 7 (evidence-based practice indicators) thresholds ---
# Tidal volume <= this (mL/kg predicted body weight, Devine formula) is the lung-protective-
# ventilation target used for both LPV indicators. ARDSnet's protocol target is 6 mL/kg; 8 mL/kg is
# used here as the more commonly reported "not clearly injurious" adherence bar in observational
# EHR studies -- change to 6 if the grant wants the stricter protocol target instead.
LPV_TARGET_ML_PER_KG_PBW = 8

# glucose (mg/dL) below this = a hypoglycemic event for the "avoidance" indicator's numerator
# (standard ICU glycemic-control threshold).
HYPOGLYCEMIA_THRESHOLD_MGDL = 70

# Step 0 — Encounter Stitching

Links related hospitalizations (e.g., ED-to-inpatient transfers, a quick readmission within a few hours of discharge) into one `encounter_block`, using [clifpy](https://github.com/Common-Longitudinal-ICU-data-Format/clifpy)'s `stitch_encounters` with a `STITCH_TIME_INTERVAL_HOURS`-hour window (6h).

Both `encounter_mapping` (`hospitalization_id` → `encounter_block`) and `adt_stitched` (ADT + `encounter_block`) are registered in DuckDB — **every step below is keyed on `encounter_block`, not `hospitalization_id`**. Raw CLIF tables remain `hospitalization_id`-keyed, so downstream steps expand `encounter_block` back out to its member hospitalizations wherever needed (via `encounter_mapping`/`adt_stitched`/`cohort_encounter_map`) — this matters at a stitch boundary, e.g. a record landing in the second hospitalization of a block before that hospitalization's own data has been charted. Step 3b (provider roster) is the exception: it stays at the `hospitalization_id`/ADT grain, since provider practice location by calendar year doesn't depend on patient-encounter stitching.

In [ ]:
# stitch_encounters expects the full hospitalization/adt tables.
hospitalization_raw = con.execute(f"SELECT * FROM '{hospitalization_path}'").df()
adt_raw = con.execute(f"SELECT * FROM '{adt_path}'").df()

hospitalization_stitched, adt_stitched, encounter_mapping = stitch_encounters(
    hospitalization_raw, adt_raw, time_interval=STITCH_TIME_INTERVAL_HOURS
)
con.register("encounter_mapping", encounter_mapping)
con.register("adt_stitched", adt_stitched)  # ADT rows + encounter_block, for block-scoped ASOF joins

n_multi = (encounter_mapping.groupby("encounter_block")["hospitalization_id"].transform("count") > 1).sum()
print(f"Hospitalizations:                 {encounter_mapping['hospitalization_id'].nunique():,}")
print(f"Encounter blocks:                 {encounter_mapping['encounter_block'].nunique():,}")
print(f"Hospitalizations stitched (block size > 1): {n_multi:,}")
encounter_mapping.head()

# Step 1 — Cohort Identification

Day 1 = the first ICU IMV record for an encounter, found via an ASOF join to the nearest-prior ADT row, filtered to `location_category = 'icu'`. Both the join and the grouping are scoped at the `encounter_block` level, so a stitch boundary can't reset the timeline. MV starting outside the ICU doesn't establish Day 1; an encounter never on IMV in an ICU location is excluded entirely. ECMO-at-onset, tracheostomy-at-onset, cardiac arrest/anoxic injury (POA), and DNI status before MV start are all evaluated at or before Day 1 and excluded, never looking forward.

In [ ]:
# --- 1a: Day 1 index -- first ICU IMV record per encounter (not per hospitalization_id) ---
# Vent->ADT ASOF match and grouping are both scoped at the encounter_block level so a stitch
# boundary can't cause a genuinely-current record to be missed (e.g. IMV starting in the second
# hospitalization of a block before that hospitalization's own ADT row is charted).
mv_first = con.execute(f"""
WITH vent_block AS (
    SELECT em.encounter_block, vp.recorded_dttm
    FROM '{vent_path}' vp
    JOIN encounter_mapping em USING (hospitalization_id)
    WHERE LOWER(vp.device_category) = 'imv'
),
all_vent_location AS (
    SELECT vb.encounter_block, vb.recorded_dttm, adt.location_category
    FROM vent_block vb
    ASOF INNER JOIN adt_stitched adt
        ON vb.encounter_block = adt.encounter_block
        AND vb.recorded_dttm >= adt.in_dttm
    WHERE LOWER(adt.location_category) = 'icu'
)
SELECT
    encounter_block,
    MIN(recorded_dttm) AS mv_start_dttm
FROM all_vent_location
GROUP BY encounter_block
""").df()
con.register("mv_first", mv_first)
print(f"Encounter blocks with >=1 ICU IMV record: {len(mv_first):,}")

In [ ]:
# --- 1b: base cohort -- aggregate hospitalization+patient to the encounter grain, apply age filter.
# admission_dttm = earliest, discharge_dttm = latest in the block; age_at_admission/discharge_category
# taken from the block's first/last hospitalization (arg_min/arg_max), mirroring clifpy's own
# block-level aggregation. hospitalization_ids is kept for traceability in the cohort output.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE encounter_hosp AS
    SELECT
        em.encounter_block,
        h.patient_id,
        MIN(h.admission_dttm) AS admission_dttm,
        MAX(h.discharge_dttm) AS discharge_dttm,
        arg_min(h.age_at_admission, h.admission_dttm) AS age_at_admission,
        arg_max(h.discharge_category, h.discharge_dttm) AS discharge_category,
        array_sort(LIST(DISTINCT h.hospitalization_id)) AS hospitalization_ids
    FROM '{hospitalization_path}' h
    JOIN encounter_mapping em USING (hospitalization_id)
    GROUP BY em.encounter_block, h.patient_id
""")

cohort_base = con.execute(f"""
    SELECT
        eh.encounter_block,
        eh.patient_id,
        eh.hospitalization_ids,
        m.mv_start_dttm,
        eh.age_at_admission,
        eh.admission_dttm,
        eh.discharge_dttm,
        eh.discharge_category,
        pt.death_dttm
    FROM mv_first m
    JOIN encounter_hosp eh USING (encounter_block)
    JOIN '{patient_path}' pt USING (patient_id)
    WHERE eh.age_at_admission >= 18
""").df()
con.register("cohort_base", cohort_base)
print(f"After age >= 18 filter: {len(cohort_base):,}")

In [ ]:
# --- 1c: ECMO-at-onset exclusion, checked across every hospitalization_id in the encounter_block ---
ecmo_onset_ids = set(con.execute(f"""
    SELECT b.encounter_block
    FROM cohort_base b
    JOIN encounter_mapping em ON em.encounter_block = b.encounter_block
    JOIN '{ecmo_path}' e ON e.hospitalization_id = em.hospitalization_id
    WHERE LOWER(e.device_category) = 'va_ecmo'
    GROUP BY b.encounter_block, b.mv_start_dttm
    HAVING MIN(e.recorded_dttm) <= b.mv_start_dttm + INTERVAL '{ECMO_ONSET_BUFFER_HOURS} hours'
""").df()["encounter_block"])
print(f"ECMO-at-onset excluded: {len(ecmo_onset_ids):,}")

In [ ]:
# --- 1d: tracheostomy-at-onset exclusion, same cross-hospitalization expansion as 1c ---
trach_onset_ids = set(con.execute(f"""
    SELECT DISTINCT b.encounter_block
    FROM cohort_base b
    JOIN encounter_mapping em ON em.encounter_block = b.encounter_block
    JOIN '{vent_path}' r ON r.hospitalization_id = em.hospitalization_id
    WHERE r.tracheostomy = 1
      AND r.recorded_dttm BETWEEN b.mv_start_dttm AND b.mv_start_dttm + INTERVAL '{TRACH_ONSET_WINDOW_HOURS} hours'
""").df()["encounter_block"])
print(f"Tracheostomy-at-onset excluded: {len(trach_onset_ids):,}")

In [ ]:
# --- 1e: DNI-before-MV exclusion ---
# code_status is patient-level, so a DNI status has no inherent tie to any one encounter -- bounded
# to [encounter_block admission, mv_start_dttm] so a status from a completely different, unrelated
# prior encounter (e.g. an earlier hospitalization years before, not part of this stitched block)
# can't incorrectly exclude the current one. Within that window, take the latest status at/before
# MV start, ranked per-encounter (a patient with multiple stitched encounters gets each its own
# correct window). ILIKE '%dni%' is a substring match, functionally equivalent to exact-matching
# DNR/DNI, DNAR/DNI, DNI_only given the known category universe (Full, DNR, DNAR, UDNR, AND,
# DNR/DNI, DNAR/DNI, DNI_only).
dni_ids = set(con.execute(f"""
    WITH ranked AS (
        SELECT b.encounter_block, dni.code_status_category,
               ROW_NUMBER() OVER (PARTITION BY b.encounter_block ORDER BY dni.start_dttm DESC) AS rn
        FROM cohort_base b
        INNER JOIN '{dnr_path}' dni USING (patient_id)
        WHERE dni.start_dttm >= b.admission_dttm AND dni.start_dttm <= b.mv_start_dttm
    )
    SELECT encounter_block
    FROM ranked
    WHERE rn = 1 AND code_status_category ILIKE '%dni%'
""").df()["encounter_block"])
print(f"DNI-before-MV excluded: {len(dni_ids):,}")

In [ ]:
# --- 1f: cardiac arrest / anoxic injury (POA) exclusion ---
# diagnosis_code formatting is inconsistent (e.g. "I46.2" vs "G931") -- REPLACE(...,'.','') normalizes
# both before matching. Checked across every hospitalization_id in the encounter (via encounter_mapping,
# not the post-exclusion cohort_encounter_map -- this exclusion hasn't been applied to cohort_base yet).
dx_check = con.execute(f"""
    SELECT b.encounter_block,
           MAX(CASE WHEN REPLACE(d.diagnosis_code, '.', '') ILIKE 'I46%' AND d.poa_present = 1
                     THEN 1 ELSE 0 END) AS cardiac_arrest,
           MAX(CASE WHEN REPLACE(d.diagnosis_code, '.', '') ILIKE 'I46%' AND d.poa_present = 1 AND d.diagnosis_primary = 1
                     THEN 1 ELSE 0 END) AS cardiac_arrest_primary,
           MAX(CASE WHEN REPLACE(d.diagnosis_code, '.', '') ILIKE 'G931%' AND d.poa_present = 1
                     THEN 1 ELSE 0 END) AS anoxic_injury,
           MAX(CASE WHEN REPLACE(d.diagnosis_code, '.', '') ILIKE 'G931%' AND d.poa_present = 1 AND d.diagnosis_primary = 1
                     THEN 1 ELSE 0 END) AS anoxic_injury_primary
    FROM cohort_base b
    JOIN encounter_mapping em ON em.encounter_block = b.encounter_block
    JOIN '{hosp_diagnosis_path}' d ON d.hospitalization_id = em.hospitalization_id
    GROUP BY b.encounter_block
""").df()
cardiac_arrest_ids = set(dx_check.loc[dx_check["cardiac_arrest"] == 1, "encounter_block"])
anoxic_injury_ids = set(dx_check.loc[dx_check["anoxic_injury"] == 1, "encounter_block"])
print(f"Cardiac arrest (POA) excluded: {len(cardiac_arrest_ids):,} "
      f"({int(dx_check['cardiac_arrest_primary'].sum()):,} coded as primary diagnosis)")
print(f"Anoxic injury (POA) excluded: {len(anoxic_injury_ids):,} "
      f"({int(dx_check['anoxic_injury_primary'].sum()):,} coded as primary diagnosis)")

In [ ]:
# --- 1g: combine all exclusions, finalize cohort ---
excluded_ids = ecmo_onset_ids | trach_onset_ids | dni_ids | cardiac_arrest_ids | anoxic_injury_ids
cohort = cohort_base[~cohort_base["encounter_block"].isin(excluded_ids)].copy()
con.register("cohort_ids", cohort[["encounter_block", "patient_id", "mv_start_dttm", "admission_dttm"]])

# cohort_encounter_map: expand the surviving cohort's encounter_blocks back to their hospitalization_id
# membership -- used everywhere downstream that pulls hospitalization_id-keyed raw-table rows.
con.execute("""
    CREATE OR REPLACE TEMP TABLE cohort_encounter_map AS
    SELECT em.hospitalization_id, em.encounter_block
    FROM encounter_mapping em
    JOIN cohort_ids c USING (encounter_block)
""")

print(f"Base cohort after all exclusions: {len(cohort):,} "
      f"(excluded {len(excluded_ids):,} of {len(cohort_base):,})")

# Step 2 — VFD-28 Computation

`x` = full days elapsed since MV start (Day 0 = initiation), `VFD28 = 28 - x` — using `floor(elapsed_hours / 24)` rather than a 1-indexed day label (a 1-indexed label would make VFD28=28 unreachable).

Reintubation is handled by walking the full device-transition sequence per encounter and keeping only the last off-support period with no subsequent reintubation before censoring. No minimum sustained-liberation duration is required (see VFD-28 Definition). Discharge censoring falls out naturally: `censor_dttm = min(mv_start + 28 days, discharge_dttm)`.

In [ ]:
# --- 2a: window / censor timestamps ---
cohort["window_end_dttm"] = cohort["mv_start_dttm"] + pd.Timedelta(days=VFD_WINDOW_DAYS)
cohort["died_in_window"] = cohort["death_dttm"].notna() & (cohort["death_dttm"] <= cohort["window_end_dttm"])
cohort["censor_dttm"] = cohort[["window_end_dttm", "discharge_dttm"]].min(axis=1)
con.register("cohort_censor", cohort[["encounter_block", "mv_start_dttm", "censor_dttm"]])

# Day-label cutoffs use the same floor-day accounting VFD-28 itself uses, so Step 4's at-risk loop
# and this step's censoring agree on which "Day d" a discharge/death instant falls into.
cohort["last_trackable_day"] = ((cohort["censor_dttm"] - cohort["mv_start_dttm"]).dt.total_seconds() // 86400) + 1

# death_dttm is date-only precision in the real data: a patient who dies at 5pm on the same
# calendar day ventilation started at 2pm still gets death_dttm = midnight of that day, which a raw
# timestamp difference would compute as BEFORE mv_start_dttm -- reading as "already dead before MV
# started" instead of "died on Day 1" (confirmed against real data: 100% of these cases cluster at
# exactly 00:00:00). death_day is recomputed here via a fresh DuckDB query rather than pandas
# arithmetic on the already-loaded cohort["death_dttm"]/["mv_start_dttm"] columns, so both get cast
# to DATE inside the same query under the connection's single SET TimeZone setting (Global
# Settings) -- this rules out the two timestamps having picked up inconsistent timezone handling
# across separate pandas operations. Comparing at the calendar-DATE level means a death on the same
# calendar date as mv_start_dttm correctly gets death_day=1, matching how Day 1 is defined
# everywhere else in this notebook. This does not affect the VFD-28 score itself (compute_vfd28
# only reads died_in_window, not death_day) -- it only affects which at-risk days a patient
# contributes to; without this fix these patients would fail is_alive on every day 1-28, not just
# Day 1, and would be silently absent from the entire at-risk series despite still contributing a
# (correctly computed) VFD28=0 to Step 5's per-day summary stats.
death_day_calc = con.execute(f"""
    SELECT c.encounter_block,
           DATE_DIFF('day', CAST(c.mv_start_dttm AS DATE), CAST(pt.death_dttm AS DATE)) + 1 AS death_day
    FROM cohort_ids c
    JOIN '{patient_path}' pt USING (patient_id)
    WHERE pt.death_dttm IS NOT NULL
""").df()
cohort = cohort.merge(death_day_calc, on="encounter_block", how="left")

# --- 2b: in-window respiratory_support records, expanded across every hospitalization_id in each
# encounter_block -- a stitched block's device-state timeline spans every hospitalization it contains.
resp_window = con.execute(f"""
    SELECT cem.encounter_block, r.recorded_dttm, r.device_category, r.tracheostomy
    FROM cohort_encounter_map cem
    JOIN '{vent_path}' r ON r.hospitalization_id = cem.hospitalization_id
    JOIN cohort_censor c ON c.encounter_block = cem.encounter_block
    WHERE r.recorded_dttm >= c.mv_start_dttm AND r.recorded_dttm <= c.censor_dttm
    ORDER BY cem.encounter_block, r.recorded_dttm
""").df()
# NIV/HFNC counts as OFF (not blocking liberation) unless the patient is tracheostomized -- see
# Pipeline Constants. .fillna(0) treats missing tracheostomy status as not-tracheostomized, matching
# the COALESCE(...,0) used in Step 4's SQL version of this same check.
device_lower = resp_window["device_category"].str.lower()
trach_known = resp_window["tracheostomy"].fillna(0)
resp_window["state"] = np.where(
    device_lower.isin(OFF_SUPPORT_DEVICES), "OFF",
    np.where(device_lower.isin(NIV_DEVICES) & (trach_known != 1), "OFF", "ON")
)
print(f"In-window respiratory_support records: {len(resp_window):,}")

In [ ]:
# --- 2c: final-liberation walk per encounter ---
def find_final_liberation(group):
    """Returns the start of the LAST off-support period with no subsequent reintubation before
    censoring -- the final extubation used to anchor VFD-28. No minimum-duration confirmation is
    required (see VFD-28 Definition). Grouped by encounter_block, not hospitalization_id, so the
    walk naturally spans a stitch boundary.
    """
    off_start = None
    for _, row in group.sort_values("recorded_dttm").iterrows():
        if row["state"] == "OFF" and off_start is None:
            off_start = row["recorded_dttm"]
        elif row["state"] == "ON" and off_start is not None:
            off_start = None  # reintubated -- this off-period doesn't count, keep looking
    return pd.Series({
        "liberated_dttm": off_start if off_start is not None else pd.NaT,
        "liberation_confirmed": off_start is not None,
    })

liberation = resp_window.groupby("encounter_block").apply(find_final_liberation).reset_index()
cohort = cohort.merge(liberation, on="encounter_block", how="left")
cohort["liberation_confirmed"] = cohort["liberation_confirmed"].fillna(False)
cohort

In [ ]:
# --- 2d: VFD-28 ---
def elapsed_full_days(mv_start, t):
    """x = number of full calendar days elapsed since initiation (Day 0 = initiation)."""
    return int(np.floor((t - mv_start).total_seconds() / 86400))

def compute_vfd28(row):
    if row["died_in_window"]:
        return 0
    if row["liberation_confirmed"]:
        x = min(elapsed_full_days(row["mv_start_dttm"], row["liberated_dttm"]), VFD_WINDOW_DAYS)
        return max(0, VFD_WINDOW_DAYS - x)
    return 0  # still ventilated at censor — no final non-reintubated extubation observed

cohort["vfd28"] = cohort.apply(compute_vfd28, axis=1)

print(cohort["vfd28"].describe())
print(f"\nProportion VFD28=0:  {(cohort['vfd28'] == 0).mean():.3f}")
print(f"Proportion VFD28=28: {(cohort['vfd28'] == 28).mean():.3f}")
print(f"Died in window:      {cohort['died_in_window'].sum():,}")
print(f"Liberation confirmed:{cohort['liberation_confirmed'].sum():,}")

# Step 3 — Provider Roster (year × ICU type)

**3a — Eligible providers:** a provider must have initiated at least `ELIGIBLE_PROVIDER_MIN_INITIATIONS` ICU IMV episodes globally to count as eligible. "Initiated" = the provider covering the patient at the exact instant of their first ICU IMV record, matched at the `encounter_block` level (the match can reach into an earlier hospitalization of the block if the later one's own provider data hasn't been charted yet). This eligible-providers set restricts both the tables below and Step 4's per-patient provider-coverage check.

**3b — Roster:** a provider is eligible for a (hospital, icu_type, year) if their provider record overlapped an ICU-location ADT interval in that year, restricted to the eligible-providers set from 3a. This table stays at the `hospitalization_id`/ADT grain — provider practice location by calendar year doesn't depend on encounter stitching. No role/title field exists in the real provider data, so "eligible" means volume-qualified, not specifically attending-level.

**3c — Daily provider availability:** the same ASOF-to-ADT resolution as 3b, but grouped by *calendar date* instead of year — for each (hospital, icu_stratum, calendar date), the count of distinct eligible providers who had ≥1 patient attributed to them that day. This is the actual daily staffing pool, not the year-long roster ceiling; attached to each at-risk patient-day by calendar date in Step 4, then summarized as median/IQR in Step 5.

In [ ]:
# --- 3a: eligible providers ---
# A provider must have initiated >= ELIGIBLE_PROVIDER_MIN_INITIATIONS ICU IMV episodes, globally
# across the whole dataset (mv_first is unfiltered), to be an "eligible provider" downstream.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE prov_hourly AS
    SELECT hospitalization_id,
           CAST(recorded_date AS TIMESTAMP) + CAST(recorded_hour AS INTEGER) * INTERVAL '1 hour' AS recorded_dttm,
           prov_npi
    FROM '{provider_path}'
    WHERE prov_npi IS NOT NULL
""")

# Expanded to the encounter_block grain so the initiating-provider match below can reach back into
# an earlier hospitalization of the same stitched block if the later one's provider data hasn't
# been charted yet at mv_start_dttm.
con.execute("""
    CREATE OR REPLACE TEMP TABLE prov_hourly_block AS
    SELECT em.encounter_block, ph.recorded_dttm, ph.prov_npi
    FROM prov_hourly ph
    JOIN encounter_mapping em USING (hospitalization_id)
""")

eligible_providers = con.execute(f"""
    WITH initiating_provider AS (
        SELECT m.encounter_block, pr.prov_npi
        FROM mv_first m
        ASOF LEFT JOIN prov_hourly_block pr
            ON m.encounter_block = pr.encounter_block
           AND m.mv_start_dttm >= pr.recorded_dttm
    )
    SELECT prov_npi, COUNT(DISTINCT encounter_block) AS n_initiations
    FROM initiating_provider
    WHERE prov_npi IS NOT NULL
    GROUP BY prov_npi
    HAVING COUNT(DISTINCT encounter_block) >= {ELIGIBLE_PROVIDER_MIN_INITIATIONS}
""").df()
con.register("eligible_providers", eligible_providers)

# eligible_prov_hourly: hospitalization_id-grain, feeds Step 3b's roster.
con.execute("""
    CREATE OR REPLACE TEMP TABLE eligible_prov_hourly AS
    SELECT ph.* FROM prov_hourly ph
    INNER JOIN eligible_providers ep ON ph.prov_npi = ep.prov_npi
""")

# eligible_prov_hourly_block: encounter_block-grain counterpart, used by Step 4.
con.execute("""
    CREATE OR REPLACE TEMP TABLE eligible_prov_hourly_block AS
    SELECT ph.* FROM prov_hourly_block ph
    INNER JOIN eligible_providers ep ON ph.prov_npi = ep.prov_npi
""")

print(f"Eligible providers (>= {ELIGIBLE_PROVIDER_MIN_INITIATIONS} lifetime ICU IMV initiations): {len(eligible_providers):,}")

In [ ]:
# --- 3b: roster (year x icu_type), restricted to eligible providers ---
provider_roster = con.sql(f"""
WITH provider_hospital AS (
    -- ASOF = nearest-prior ADT row (current physical location at that hour), not an exact
    -- DATE+HOUR match, since real ADT transfers happen only a handful of times per stay.
    -- hospital_type/hospital_id filtered in WHERE, after resolving current location, so the join
    -- can't skip a genuinely-current non-matching row to find an older matching one.
    SELECT
        ph.hospitalization_id,
        ph.recorded_dttm,
        ph.prov_npi,
        YEAR(ph.recorded_dttm) AS year,
        adt.hospital_id,
        adt.hospital_type,
        adt.location_type
    FROM eligible_prov_hourly ph
    ASOF LEFT JOIN '{adt_path}' adt
        ON ph.hospitalization_id = adt.hospitalization_id
       AND ph.recorded_dttm >= adt.in_dttm
    WHERE adt.hospital_type IN ('community', 'academic')
      AND adt.hospital_id != 'Missing'
), community_hospitals AS (
    SELECT
        year,
        hospital_id,
        ANY_VALUE(hospital_type) as hospital_type,
        NULL as location_type,
        LIST(distinct prov_npi) as prov_npi_list,
        COUNT(distinct prov_npi) as prov_npi_count
    FROM provider_hospital
    WHERE hospital_type = 'community'
    GROUP BY year, hospital_id
), academic_hospitals AS (
    SELECT
        year,
        hospital_id,
        ANY_VALUE(hospital_type) as hospital_type,
        location_type,
        LIST(distinct prov_npi) as prov_npi_list,
        COUNT(distinct prov_npi) as prov_npi_count
    FROM provider_hospital
    WHERE hospital_type = 'academic'
    GROUP BY year, hospital_id, location_type 
)
SELECT *
FROM community_hospitals
UNION ALL
SELECT *
FROM academic_hospitals

ORDER BY year, hospital_id, location_type
""").df()
provider_roster

In [ ]:
# --- 3c: daily provider availability (calendar-date grain, not relative VFD day) ---
# Same ASOF-to-ADT resolution as 3b, grouped by calendar date instead of year: for each
# (hospital, icu_stratum, calendar_date), the count of distinct ELIGIBLE providers
# (eligible_prov_hourly, same >= ELIGIBLE_PROVIDER_MIN_INITIATIONS bar as everywhere else) who had
# >=1 patient attributed to them that day. This is the actual daily staffing pool a patient could
# have drawn from -- distinct from 3b's year-long roster, which only bounds the annual maximum.
# calendar_date is cast to a plain VARCHAR date string (not left as a DATE/TIMESTAMP) because
# DuckDB's date_trunc()/DATE-cast can convert to pandas as a naive datetime64, while the merge
# partner on the pandas side (Step 4g) is built from a tz-aware column -- a str/str comparison
# sidesteps that whole category of dtype mismatch rather than trying to keep two independently
# timezone-handled datetime64 columns in sync (the same class of bug death_day had, Step 2a).
provider_daily = con.sql(f"""
WITH provider_hospital AS (
    SELECT
        ph.prov_npi,
        CAST(CAST(ph.recorded_dttm AS DATE) AS VARCHAR) AS calendar_date,
        adt.hospital_id,
        adt.hospital_type,
        adt.location_type
    FROM eligible_prov_hourly ph
    ASOF LEFT JOIN '{adt_path}' adt
        ON ph.hospitalization_id = adt.hospitalization_id
       AND ph.recorded_dttm >= adt.in_dttm
    WHERE adt.hospital_type IN ('community', 'academic')
      AND adt.hospital_id != 'Missing'
), community_hospitals AS (
    SELECT
        calendar_date,
        hospital_id,
        ANY_VALUE(hospital_type) as hospital_type,
        NULL as location_type,
        COUNT(distinct prov_npi) as n_providers_that_day
    FROM provider_hospital
    WHERE hospital_type = 'community'
    GROUP BY calendar_date, hospital_id
), academic_hospitals AS (
    SELECT
        calendar_date,
        hospital_id,
        ANY_VALUE(hospital_type) as hospital_type,
        location_type,
        COUNT(distinct prov_npi) as n_providers_that_day
    FROM provider_hospital
    WHERE hospital_type = 'academic'
    GROUP BY calendar_date, hospital_id, location_type
)
SELECT * FROM community_hospitals
UNION ALL
SELECT * FROM academic_hospitals
""").df()
print(f"Daily provider-availability rows (hospital x stratum x calendar date): {len(provider_daily):,}")
print(f"Providers-per-day, overall distribution -- median: {provider_daily['n_providers_that_day'].median():.1f}, "
      f"IQR: [{provider_daily['n_providers_that_day'].quantile(0.25):.1f}, {provider_daily['n_providers_that_day'].quantile(0.75):.1f}]")

# Step 4 — Daily At-Risk Landmark Loop (Day 1–28)

For each encounter × each Day *d*, at-risk requires: alive, not yet discharged, still on the ventilator (real-time device state), still in an ICU location, and covered by an eligible provider — evaluated at the start of Day *d* (`mv_start_dttm + (d-1) days`).

VFD-28 (Step 2) only cares about the *final* liberation. The at-risk "still on the ventilator" check needs the actual moment-to-moment device state, including brief off-periods that later get reintubated — these don't count toward VFD-28, but do mean the patient is genuinely off the vent (not at-risk) for those days. DuckDB's ASOF JOIN finds the most recent device-state record at or before each day's start.

In [ ]:
# --- 4a: real-time device-state timeline (distinct from Step 2's "final liberation" logic) ---
# Expanded across every hospitalization_id in each encounter_block, same as Step 2b's resp_window.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE resp_state AS
    SELECT cem.encounter_block, r.recorded_dttm,
           CASE
               WHEN LOWER(r.device_category) IN {tuple(OFF_SUPPORT_DEVICES)} THEN 'OFF'
               -- COALESCE(...,0): raw tracheostomy != 1 is NULL (not TRUE) when missing, which
               -- would otherwise fall through to ON instead of OFF -- keeps this in agreement with
               -- the pandas version in Step 2b for the same missing-value case.
               WHEN LOWER(r.device_category) IN {tuple(NIV_DEVICES)} AND COALESCE(r.tracheostomy, 0) != 1 THEN 'OFF'
               ELSE 'ON'
           END AS state
    FROM cohort_encounter_map cem
    JOIN '{vent_path}' r ON r.hospitalization_id = cem.hospitalization_id
    JOIN cohort_censor c ON c.encounter_block = cem.encounter_block
    WHERE r.recorded_dttm >= c.mv_start_dttm AND r.recorded_dttm <= c.censor_dttm
""")

In [ ]:
# --- 4b: cohort x Day 1..28 scaffold ---
con.register("cohort_full", cohort)
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE days AS
    SELECT c.encounter_block, c.patient_id, c.mv_start_dttm, c.death_dttm,
           c.discharge_dttm, c.censor_dttm, c.vfd28, c.last_trackable_day, c.death_day,
           d.day,
           -- fixed-duration 24h steps, not calendar 'INTERVAL 1 day' (DST-aware), to stay aligned
           -- with Step 2's fixed pd.Timedelta(days=28) window.
           c.mv_start_dttm + (d.day - 1) * INTERVAL '24 hours' AS day_start_dttm
    FROM cohort_full c
    CROSS JOIN generate_series(1, {VFD_WINDOW_DAYS}) AS d(day)
""")
print(f"Day-level rows (cohort x {VFD_WINDOW_DAYS} days): {con.execute('SELECT count(*) FROM days').fetchone()[0]:,}")

In [ ]:
# --- 4c: ASOF join for real-time on/off state at each day's start ---
con.execute("""
    CREATE OR REPLACE TEMP TABLE days_state AS
    SELECT d.*, COALESCE(rs.state, 'ON') AS state_at_day_start
    FROM days d
    ASOF LEFT JOIN resp_state rs
      ON d.encounter_block = rs.encounter_block
     AND d.day_start_dttm >= rs.recorded_dttm
""")

In [ ]:
# provider is sparse hourly-snapshot data -- ASOF (nearest-prior-hour) lookup, same reasoning as
# the ADT join below: an exact DATE+HOUR match would silently drop coverage on any hour with no
# recorded provider row. Uses eligible_prov_hourly_block (Step 3a, encounter_block-grain).
daily = con.sql(f"""
    SELECT
        ds.encounter_block,
        ds.patient_id,
        ds.day,
        ds.day_start_dttm,
        ds.vfd28,
        adt.hospital_id,
        adt.hospital_type,
        adt.location_type,
        -- Stratification key: per-unit for academic (East Bank) sites, pooled (NULL) for community
        -- sites -- mirrors provider_roster (Step 3) so the merge in 4f and Step 5's GROUP BY line up.
        CASE WHEN adt.hospital_type = 'academic' THEN adt.location_type ELSE NULL END AS icu_stratum,
        YEAR(ds.mv_start_dttm) AS index_year,
        -- Compares integer "Day d" labels (Step 2's floor-day accounting), not raw timestamps, so
        -- this boundary stays consistent with Step 2's censoring.
        (ds.death_day IS NULL OR ds.day <= ds.death_day) AS is_alive,
        (ds.day <= ds.last_trackable_day) AS not_discharged,
        (ds.state_at_day_start = 'ON') AS still_on_vent,
        -- location_type is NULL for every non-ICU location and populated only for ICU rows, so
        -- "location_type IS NOT NULL" is exactly "currently in an ICU." hospital_id = 'Missing'
        -- (a data-quality artifact) is excluded after the ASOF join resolves the true current
        -- location, not by pre-filtering ADT before joining.
        (adt.location_type IS NOT NULL AND adt.hospital_id != 'Missing') AS still_in_unit,
        (pr.prov_npi IS NOT NULL) AS has_provider
    FROM days_state ds
    -- ASOF = most recent ADT row at or before this hour, scoped at the encounter_block level via
    -- adt_stitched so the lookup can reach back into an earlier hospitalization of the same
    -- stitched block. Not an exact DATE+HOUR match: the patient stays at a location until their
    -- next transfer, so the most recent transfer-in row at or before day_start_dttm is current.
    ASOF LEFT JOIN adt_stitched adt
        ON ds.encounter_block = adt.encounter_block
       AND ds.day_start_dttm >= adt.in_dttm
    ASOF LEFT JOIN eligible_prov_hourly_block pr
        ON ds.encounter_block = pr.encounter_block
       AND ds.day_start_dttm >= pr.recorded_dttm
""").df()
daily

In [ ]:
daily["at_risk"] = (
    daily["is_alive"] & daily["not_discharged"] & daily["still_on_vent"]
    & daily["still_in_unit"].fillna(False) & daily["has_provider"].fillna(False)
)

# --- 4f: attach per-patient eligible-provider count from the roster (indexed at MV-start year) ---
# Merged on icu_stratum, not raw location_type -- provider_roster's community rows have
# location_type=NULL (pooled), which icu_stratum already mirrors.
daily = daily.merge(
    provider_roster[["hospital_id", "location_type", "year", "prov_npi_count"]].rename(
        columns={"year": "index_year", "location_type": "icu_stratum", "prov_npi_count": "n_eligible_providers"}
    ),
    on=["hospital_id", "icu_stratum", "index_year"], how="left",
)

# --- 4g: attach the daily (calendar-date) eligible-provider count from provider_daily (Step 3c) --
# calendar_date is the actual date each patient's own VFD day fell on -- not the relative VFD day
# itself, which almost never lines up across patients. Formatted as a plain 'YYYY-MM-DD' string
# (not left as a datetime64) to match provider_daily's VARCHAR calendar_date (Step 3c) -- a
# str/str comparison, immune to whether day_start_dttm ends up timezone-aware or naive, rather than
# trying to keep two independently timezone-handled datetime64 columns in sync (a real failure
# mode: day_start_dttm can convert to pandas as tz-aware while a DuckDB date_trunc()/DATE-cast
# column converts as naive, which pandas .merge() refuses outright rather than silently misjoining).
daily["calendar_date"] = daily["day_start_dttm"].dt.strftime("%Y-%m-%d")
daily = daily.merge(
    provider_daily[["hospital_id", "location_type", "calendar_date", "n_providers_that_day"]].rename(
        columns={"location_type": "icu_stratum", "n_providers_that_day": "n_eligible_providers_daily"}
    ),
    on=["hospital_id", "icu_stratum", "calendar_date"], how="left",
)

print(f"at_risk rows: {daily['at_risk'].sum():,} / {len(daily):,}")
daily.groupby("day")["at_risk"].sum()

In [ ]:
# --- Diagnostic: why do patients drop off the at-risk set each day? ---
# For every encounter_block x day, was it at-risk the PREVIOUS day but not this day? (day=1's
# "previous day" is treated as True by convention, so day-1 dropouts -- e.g. no eligible provider
# yet -- are captured the same way as every later day's dropouts.)
daily_sorted = daily.sort_values(["encounter_block", "day"]).copy()
daily_sorted["prev_at_risk"] = daily_sorted.groupby("encounter_block")["at_risk"].shift(1)
daily_sorted.loc[daily_sorted["day"] == 1, "prev_at_risk"] = True

dropped = daily_sorted[(daily_sorted["prev_at_risk"] == True) & (~daily_sorted["at_risk"])].copy()
for col in ["still_in_unit", "has_provider"]:
    dropped[col] = dropped[col].fillna(False)

conditions = ["is_alive", "not_discharged", "still_on_vent", "still_in_unit", "has_provider"]
dropped["reason"] = dropped[conditions].apply(
    lambda row: ", ".join(c for c in conditions if not row[c]), axis=1
)

print("Total dropped per day (should roughly match the day-over-day decline in your at_risk counts):")
print(dropped.groupby("day").size().to_string())
print()

print("Reason breakdown per day (exact combination of failed conditions, not mutually exclusive across rows):")
reason_by_day = dropped.groupby(["day", "reason"]).size().reset_index(name="n_dropped")
print(reason_by_day.to_string(index=False))
print()

print("Reason totals across ALL days pooled (which single factor drives the most attrition overall):")
print(dropped["reason"].value_counts().to_string())


# Step 5 — Daily Summary Aggregation & Output

One row per (day, hospital, icu_type, year), computed over that day's at-risk encounters. `suppressed_lt10` flags strata with n_at_risk < 10 (CLIF federated small-cell convention) without removing them from the aggregate outputs.

In [ ]:
at_risk = daily[daily["at_risk"]].copy()
con.register("at_risk", at_risk)

daily_summary = con.execute("""
    SELECT
        index_year AS year, hospital_id, icu_stratum, day,
        COUNT(*) AS n_at_risk,
        AVG(vfd28) AS vfd28_mean,
        STDDEV(vfd28) AS vfd28_sd,
        MEDIAN(vfd28) AS vfd28_median,
        QUANTILE_CONT(vfd28, 0.25) AS vfd28_q1,
        QUANTILE_CONT(vfd28, 0.75) AS vfd28_q3,
        AVG(CASE WHEN vfd28 = 0 THEN 1.0 ELSE 0 END) AS vfd28_prop_0,
        AVG(CASE WHEN vfd28 = 28 THEN 1.0 ELSE 0 END) AS vfd28_prop_28,
        MAX(n_eligible_providers) AS elig_providers_roster_n,
        MEDIAN(n_eligible_providers_daily) AS elig_providers_daily_median,
        QUANTILE_CONT(n_eligible_providers_daily, 0.25) AS elig_providers_daily_q1,
        QUANTILE_CONT(n_eligible_providers_daily, 0.75) AS elig_providers_daily_q3
    FROM at_risk
    GROUP BY index_year, hospital_id, icu_stratum, day
    ORDER BY index_year, hospital_id, icu_stratum, day
""").df()

daily_summary["suppressed_lt10"] = daily_summary["n_at_risk"] < 10
print(f"Stratum-days with n_at_risk < 10: {daily_summary['suppressed_lt10'].sum()} / {len(daily_summary)}")

# --- write outputs (CSV) ---
# cohort is patient-level (patient_id, dates, hospitalization_ids) -- stays no_share only.
cohort.drop(columns=["window_end_dttm"]).to_csv("output_no_share/cohort.csv", index=False)
provider_roster.to_csv("output_to_box/provider_roster.csv", index=False)

# daily_summary is a pure aggregate (no patient_id, no dates), so the full unsuppressed table goes
# to output_to_box; suppressed_lt10 flags small-N rows without physically removing them.
daily_summary.to_csv("output_to_box/daily_summary.csv", index=False)
daily_summary.to_csv("output_no_share/daily_summary_unsuppressed.csv", index=False)

print("\nWrote: output_no_share/cohort.csv, output_no_share/daily_summary_unsuppressed.csv")
print("Wrote: output_to_box/provider_roster.csv, output_to_box/daily_summary.csv (unsuppressed, no PHI)")
daily_summary

# Step 6 — QC Summary

Every diagnostic count/statistic from Steps 0–4 (cohort sizes, exclusion counts, VFD-28 summary statistics, eligible-provider count, and the daily at-risk-by-day breakdown), collected into one `metric, value` table. Pure aggregate counts/statistics — no `patient_id`, no individual timestamps.

In [ ]:
at_risk_by_day = daily.groupby("day")["at_risk"].sum()

qc_rows = [
    ("n_hospitalizations", encounter_mapping["hospitalization_id"].nunique()),
    ("n_encounter_blocks", encounter_mapping["encounter_block"].nunique()),
    ("n_hospitalizations_stitched", n_multi),
    ("n_encounter_blocks_with_icu_imv", len(mv_first)),
    ("n_ecmo_onset_excluded", len(ecmo_onset_ids)),
    ("n_trach_onset_excluded", len(trach_onset_ids)),
    ("n_dni_excluded", len(dni_ids)),
    ("n_cardiac_arrest_excluded", len(cardiac_arrest_ids)),
    ("n_anoxic_injury_excluded", len(anoxic_injury_ids)),
    ("n_base_cohort_after_exclusions", len(cohort)),
    ("n_resp_window_records", len(resp_window)),
    ("vfd28_count", cohort["vfd28"].count()),
    ("vfd28_mean", cohort["vfd28"].mean()),
    ("vfd28_std", cohort["vfd28"].std()),
    ("vfd28_min", cohort["vfd28"].min()),
    ("vfd28_25pct", cohort["vfd28"].quantile(0.25)),
    ("vfd28_50pct", cohort["vfd28"].quantile(0.50)),
    ("vfd28_75pct", cohort["vfd28"].quantile(0.75)),
    ("vfd28_max", cohort["vfd28"].max()),
    ("vfd28_prop_0", (cohort["vfd28"] == 0).mean()),
    ("vfd28_prop_28", (cohort["vfd28"] == 28).mean()),
    ("n_died_in_window", int(cohort["died_in_window"].sum())),
    ("n_liberation_confirmed", int(cohort["liberation_confirmed"].sum())),
    ("n_eligible_providers", len(eligible_providers)),
    ("n_provider_daily_rows", len(provider_daily)),
    ("providers_per_day_median", provider_daily["n_providers_that_day"].median()),
    ("providers_per_day_25pct", provider_daily["n_providers_that_day"].quantile(0.25)),
    ("providers_per_day_75pct", provider_daily["n_providers_that_day"].quantile(0.75)),
] + [(f"at_risk_day_{d}", int(n)) for d, n in at_risk_by_day.items()]

qc_summary = pd.DataFrame(qc_rows, columns=["metric", "value"])
qc_summary.to_csv("output_to_box/qc_summary.csv", index=False)
print(f"Wrote: output_to_box/qc_summary.csv ({len(qc_summary)} metrics)")
qc_summary

# Step 7 — Evidence-Based Practice Indicators

Pooled adherence rates for five evidence-based ventilator-management practices, over the same VFD-28 analytic cohort (`cohort_ids`) — not stratified by hospital/year/day, just one pooled `(n_applicable, n_delivered)` pair per indicator, as needed for Aim 1's power calculation. All four implemented indicators use the same Day 1..28 structure as VFD-28 itself (fixed 24h windows from `mv_start_dttm`, Step 4's `days` scaffold — not calendar-date boundaries). Three of them use Step 5's `at_risk` table as their day population; proning uses its own IMV-day population instead (see below) since it isn't gated by provider-coverage/ICU-location, which are provider-attribution concepts irrelevant to whether proning happened.

- **Proning** — `ards_classifier.R` (Hochberg et al., PROSEVA-based, run as a subprocess exactly as in `ltvv_wrangler.ipynb`) is used ONLY to identify which encounters are ARDS-eligible (joined back on plain `hospitalization_id`, not the R script's own internal 3h-window encounter numbering, and restricted to encounters that also survive this notebook's Step 1 exclusions). Applicable = every Day 1..28 of those encounters (reusing Step 4's `days` scaffold directly) with ≥1 IMV device record in that day's window. Delivered = the subset of those days with a `position_category='prone'` event in the same window.
- **LPV, first 24 hours** — one row per patient (not per day), restricted to patients who were actually at risk on Day 1: the latest tidal-volume reading within the first 24 hours (mL/kg predicted body weight, Devine formula), target ≤ `LPV_TARGET_ML_PER_KG_PBW`.
- **LPV, subsequent days** — same target and same "latest reading in the day" rule, per at-risk patient-day from Day 2 onward.
- **Hypoglycemia avoidance** — applicable to every at-risk patient-day; delivered = no glucose < `HYPOGLYCEMIA_THRESHOLD_MGDL` recorded that day. **Limitation:** a day with no glucose drawn at all is counted as "avoided" by default — this indicator cannot distinguish a genuinely normal glucose day from a day where glucose simply wasn't measured.
- **SBT (spontaneous breathing trial) — not implemented.** CLIF has no obvious field for this: it isn't one of `patient_assessments`' categories (Sedation/Neurological/Pain/Withdrawal) and there's no ventilator-mode-based SBT flag. Would need a data-source decision before this can be added.

**`lab_category` value below (`glucose`) is a best guess, not confirmed against this site's mCIDE** — 7d prints the distinct `lab_category` values actually present so a mismatch is caught immediately (a wrong category string fails silently otherwise: zero matches, not an error).

In [ ]:
# --- 7a: predicted body weight (PBW) per encounter, Devine formula ---
# height = median of all height_cm vitals recorded for the encounter (robust to a rare outlier;
# adult height doesn't meaningfully change over an ICU stay). PBW is NULL when height or
# sex_category is missing/unrecognized -- those encounters are simply excluded from the LPV
# indicators' denominators below (missing data -> not evaluable, not assumed compliant or not).
patient_pbw = con.execute(f"""
    WITH height AS (
        SELECT cem.encounter_block, MEDIAN(v.vital_value) AS height_cm
        FROM cohort_encounter_map cem
        JOIN '{vitals_path}' v ON v.hospitalization_id = cem.hospitalization_id
        WHERE LOWER(v.vital_category) = 'height_cm' AND v.vital_value IS NOT NULL
        GROUP BY cem.encounter_block
    )
    SELECT c.encounter_block, h.height_cm, pt.sex_category,
           CASE
               WHEN h.height_cm IS NULL THEN NULL
               WHEN LOWER(pt.sex_category) = 'male'   THEN 50.0 + 2.3 * ((h.height_cm / 2.54) - 60)
               WHEN LOWER(pt.sex_category) = 'female' THEN 45.5 + 2.3 * ((h.height_cm / 2.54) - 60)
               ELSE NULL
           END AS pbw_kg
    FROM cohort_ids c
    JOIN '{patient_path}' pt USING (patient_id)
    LEFT JOIN height h ON h.encounter_block = c.encounter_block
""").df()
con.register("patient_pbw", patient_pbw)
print(f"Predicted body weight computed for {patient_pbw['pbw_kg'].notna().sum():,} / {len(patient_pbw):,} encounters")

In [ ]:
# --- 7b: shared respiratory-settings pull (tidal_volume), encounter_block-grain ---
# Feeds both LPV indicators (7d, 7e). _obs preferred over _set. fio2/peep dropped -- they were
# only ever needed for the homegrown P/F-ratio ARDS proxy, which 7c no longer uses now that ARDS
# eligibility comes from ards_classifier.R instead.
resp_settings = con.execute(f"""
    SELECT cem.encounter_block, r.recorded_dttm,
           COALESCE(r.tidal_volume_obs, r.tidal_volume_set) AS tidal_volume
    FROM cohort_encounter_map cem
    JOIN '{vent_path}' r ON r.hospitalization_id = cem.hospitalization_id
""").df()
con.register("resp_settings", resp_settings)
print(f"Respiratory-setting records pulled: {len(resp_settings):,}")

In [ ]:
# --- 7c: proning, via ards_classifier.R (Hochberg et al., PROSEVA-based AHRF classifier) ---
# ards_classifier.R is used ONLY to identify which encounters are ARDS-eligible (its
# cohort_eligible flag, net of its own ECMO/OR-before-enrollment/cardiac-arrest exclusions). It
# builds its own encounter linking internally (a 3h admission/discharge window, distinct from this
# notebook's own 6h encounter_block) -- we never rely on its internal encounter_block numbering
# matching ours, joining back on the plain hospitalization_id instead. Everything past "which
# encounters are ARDS-eligible" (the day-level applicable/delivered counts below) is computed here,
# not in R: n_applicable = every Day 1..28 of those encounters (Step 4's own "days" scaffold --
# fixed 24h windows from mv_start_dttm, the same VFD-28 day structure used by every other day-level
# computation in this notebook, not calendar-date boundaries) with >=1 IMV device_category record
# in that day's window; n_delivered = the subset of those days with a position_category='prone'
# event in the same window.
ards_output_dir = "output_no_share/ards_classifier"
os.makedirs(ards_output_dir, exist_ok=True)

# ards_classifier.R now reads this instead of scanning the whole site's respiratory_support table
# itself (that scan used to run inside R via duckdb, keyed on a nonstandard mdm_link_id column that
# isn't part of CLIF's respiratory_support schema -- removed along with the duckdb/DBI dependency
# entirely). This scopes the R script to exactly this notebook's own analytic cohort, already
# confirmed IMV-eligible in Step 1, so it never needs to touch respiratory_support for that purpose.
cohort_hosp_ids_for_r = con.execute("""
    SELECT DISTINCT cem.hospitalization_id, c.patient_id
    FROM cohort_encounter_map cem
    JOIN cohort_ids c ON c.encounter_block = cem.encounter_block
""").df()
cohort_hosp_ids_for_r.to_csv("cohort_hospitalization_ids.csv", index=False)
print(f"Wrote cohort_hospitalization_ids.csv for ards_classifier.R: {len(cohort_hosp_ids_for_r):,} hospitalization_ids")

# config.json's paths.clif is what ards_classifier.R actually reads; keep it in lockstep with this
# notebook's own data_directory rather than maintaining two independently-edited copies of the path.
with open("config.json", "r", encoding="utf-8-sig") as f:
    _cfg_for_r = json.load(f)
_cfg_for_r.setdefault("paths", {})["clif"] = clif_path
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(_cfg_for_r, f, indent=2)

# run_ards.bat falls back to whatever "Rscript" resolves to on PATH if RSCRIPT isn't set, which
# may not be the R install with the packages this script needs -- config.json's rscript_path (site-
# specific, gitignored) pins it explicitly when set, matching run_ards.bat's own RSCRIPT convention.
if sys.platform == "win32":
    _r_env = os.environ.copy()
    if _cfg_for_r.get("rscript_path"):
        _r_env["RSCRIPT"] = _cfg_for_r["rscript_path"]
    _r_cmd, _r_run_env = ["cmd", "/c", "run_ards.bat"], _r_env
else:
    _r_cmd, _r_run_env = ["Rscript", "ards_classifier.R"], None

# Explicitly captured rather than left to inherit the console (check=True alone) -- on Windows, a
# subprocess launched from a Jupyter kernel with no attached console can have its stdout/stderr
# silently dropped instead of reaching the cell output, so a real R-side failure shows up as nothing
# but a bare CalledProcessError with no clue why. Printing it ourselves guarantees it's visible.
_r_result = subprocess.run(_r_cmd, cwd=os.getcwd(), env=_r_run_env, capture_output=True, text=True)
if _r_result.stdout:
    print("--- ards_classifier.R stdout ---")
    print(_r_result.stdout)
if _r_result.stderr:
    print("--- ards_classifier.R stderr ---")
    print(_r_result.stderr)
if _r_result.returncode != 0:
    if os.path.exists("status.jsonl"):
        print("--- status.jsonl (progress before failure) ---")
        with open("status.jsonl", "r", encoding="utf-8") as f:
            print(f.read())
    # cohort_hospitalization_ids.csv is patient-level and was already written before the R call
    # regardless of how far it got -- move it out of the project root even on failure, same as the
    # success path below does for everything else.
    if os.path.exists("cohort_hospitalization_ids.csv"):
        shutil.move("cohort_hospitalization_ids.csv", f"{ards_output_dir}/cohort_hospitalization_ids.csv")
    raise RuntimeError(f"ards_classifier.R failed (exit code {_r_result.returncode}) -- see output above.")

# ards_classifier.R writes these to the working directory; relocate into output_no_share (patient-
# level -- must never go to output_to_box) rather than leaving them in the project root.
for fname in ["ards_classifier_cohort.csv", "status.jsonl", "all_hosp_ids_on_vent.parquet",
              "cohort_hospitalization_ids.csv"]:
    if os.path.exists(fname):
        shutil.move(fname, f"{ards_output_dir}/{fname}")

ards_cohort = pd.read_csv(f"{ards_output_dir}/ards_classifier_cohort.csv", index_col=0,
                          dtype={"hospitalization_id": str})
print(f"ARDS classifier rows loaded: {len(ards_cohort):,}")

# The R script's own final join (vent_eligible left-joined with ards_reviewed_merge, which is
# built from a hospitalization_id-grain table but keyed on (patient_id, encounter_block)) can
# duplicate a row when an encounter_block links >1 hospitalization_id with different
# cardiac_arrest_primary_dx values -- ltvv_wrangler.ipynb hits and documents this same quirk
# ("duplicate hospitalizations_joined_id... only difference is cardiac_arrest_primary_dx").
# Collapse it the same way: if ANY duplicate row for a hospitalization_id was flagged
# cardiac_arrest_primary_dx=1, keep that one (exclude), not the coincidental 0 duplicate.
n_before = len(ards_cohort)
ards_cohort = (ards_cohort.sort_values("cardiac_arrest_primary_dx", ascending=False)
                          .drop_duplicates("hospitalization_id"))
if n_before != len(ards_cohort):
    print(f"Collapsed {n_before - len(ards_cohort):,} duplicate ARDS-classifier rows "
          f"(same hospitalization_id, differing only in cardiac_arrest_primary_dx)")
con.register("ards_cohort", ards_cohort)

# ARDS-eligible encounters = the R script's cohort_eligible flag, net of its own ECMO/OR-before-
# enrollment/cardiac-arrest exclusions, restricted to this notebook's own cohort_ids (Step 1
# exclusions apply IN ADDITION -- the two exclusion sets are related but not identical).
ards_eligible_blocks = con.execute("""
    SELECT DISTINCT cem.encounter_block
    FROM ards_cohort ac
    JOIN cohort_encounter_map cem ON CAST(cem.hospitalization_id AS VARCHAR) = CAST(ac.hospitalization_id AS VARCHAR)
    JOIN cohort_ids c ON c.encounter_block = cem.encounter_block
    WHERE ac.cohort_eligible = 1 AND ac.ecmo_exclusion = 0
      AND ac.or_before_enrollment = 0 AND ac.cardiac_arrest_primary_dx = 0
""").df()
con.register("ards_eligible_blocks", ards_eligible_blocks)
print(f"ARDS-eligible encounters (in this notebook's cohort): {len(ards_eligible_blocks):,}")

# n_applicable = every (encounter_block, day) from Step 4's "days" scaffold (Day 1..28, fixed 24h
# windows from mv_start_dttm) with >=1 IMV device record in that day's window. n_delivered = the
# subset of those days with >=1 position_category='prone' event in the same window. Reuses "days"
# directly rather than re-deriving day boundaries, so this can never drift out of sync with how
# every other Day-N computation in this notebook defines a day.
proning = con.execute(f"""
    WITH imv_days AS (
        SELECT DISTINCT d.encounter_block, d.day, d.day_start_dttm
        FROM ards_eligible_blocks aeb
        JOIN days d ON d.encounter_block = aeb.encounter_block
        JOIN cohort_encounter_map cem ON cem.encounter_block = d.encounter_block
        JOIN '{vent_path}' r ON r.hospitalization_id = cem.hospitalization_id
            AND r.recorded_dttm >= d.day_start_dttm AND r.recorded_dttm < d.day_start_dttm + INTERVAL '24 hours'
        WHERE LOWER(r.device_category) = 'imv'
    ),
    proned_days AS (
        SELECT DISTINCT id.encounter_block, id.day
        FROM imv_days id
        JOIN cohort_encounter_map cem ON cem.encounter_block = id.encounter_block
        JOIN '{position_path}' p ON p.hospitalization_id = cem.hospitalization_id
            AND p.recorded_dttm >= id.day_start_dttm AND p.recorded_dttm < id.day_start_dttm + INTERVAL '24 hours'
        WHERE LOWER(p.position_category) = 'prone'
    )
    SELECT
        (SELECT COUNT(*) FROM imv_days) AS n_applicable,
        (SELECT COUNT(*) FROM proned_days) AS n_delivered
""").df()
print(f"\nProning -- ARDS-eligible IMV patient-days: {int(proning['n_applicable'].iloc[0]):,}, "
      f"proned: {int(proning['n_delivered'].iloc[0]):,}")

In [ ]:
# --- 7d: LPV, first 24 hours (one row per patient) ---
# Tidal volume nearest (at or before) the 24h mark -- arg_max(tidal_volume, recorded_dttm) within
# the window picks the LATEST reading in [mv_start_dttm, mv_start_dttm+24h), i.e. closest to 24h.
# Restricted to encounters that were actually AT RISK on Day 1 (not all of cohort_ids) -- keeps this
# indicator's denominator consistent with proning/LPV-subsequent-days/hypoglycemia, which all draw
# from at_risk. Without this, a patient excluded from Day-1 at-risk (e.g. no eligible provider yet)
# would still count here while being excluded from every other indicator, breaking "same cohort as
# the VFD-28 file" for this one indicator specifically.
lpv_first24h = con.execute(f"""
    WITH day1_at_risk AS (
        SELECT DISTINCT encounter_block FROM at_risk WHERE day = 1
    ),
    tv_at_24h AS (
        SELECT c.encounter_block,
               arg_max(rs.tidal_volume, rs.recorded_dttm) AS tv_near_24h
        FROM cohort_ids c
        JOIN day1_at_risk d1 ON d1.encounter_block = c.encounter_block
        JOIN resp_settings rs ON rs.encounter_block = c.encounter_block
            AND rs.recorded_dttm >= c.mv_start_dttm
            AND rs.recorded_dttm < c.mv_start_dttm + INTERVAL '24 hours'
            AND rs.tidal_volume IS NOT NULL
        GROUP BY c.encounter_block
    )
    SELECT
        COUNT(*) AS n_applicable,
        SUM(CASE WHEN t.tv_near_24h <= {LPV_TARGET_ML_PER_KG_PBW} * pbw.pbw_kg THEN 1 ELSE 0 END) AS n_delivered
    FROM tv_at_24h t
    JOIN patient_pbw pbw ON pbw.encounter_block = t.encounter_block
    WHERE pbw.pbw_kg IS NOT NULL
""").df()
print(f"\nLPV, first 24h -- evaluable patients: {int(lpv_first24h['n_applicable'].iloc[0]):,}, "
      f"met target: {int(lpv_first24h['n_delivered'].iloc[0]):,}")

In [ ]:
# --- 7e: LPV, subsequent days (per at-risk patient-day, Day 2 onward) ---
lpv_subsequent = con.execute(f"""
    WITH day_tv AS (
        SELECT ar.encounter_block, ar.day,
               arg_max(rs.tidal_volume, rs.recorded_dttm) AS tv_that_day
        FROM at_risk ar
        JOIN resp_settings rs ON rs.encounter_block = ar.encounter_block
            AND rs.recorded_dttm >= ar.day_start_dttm
            AND rs.recorded_dttm < ar.day_start_dttm + INTERVAL '24 hours'
            AND rs.tidal_volume IS NOT NULL
        WHERE ar.day >= 2
        GROUP BY ar.encounter_block, ar.day
    )
    SELECT
        COUNT(*) AS n_applicable,
        SUM(CASE WHEN dt.tv_that_day <= {LPV_TARGET_ML_PER_KG_PBW} * pbw.pbw_kg THEN 1 ELSE 0 END) AS n_delivered
    FROM day_tv dt
    JOIN patient_pbw pbw ON pbw.encounter_block = dt.encounter_block
    WHERE pbw.pbw_kg IS NOT NULL
""").df()
print(f"LPV, subsequent days -- evaluable patient-days: {int(lpv_subsequent['n_applicable'].iloc[0]):,}, "
      f"met target: {int(lpv_subsequent['n_delivered'].iloc[0]):,}")

In [ ]:
# --- 7f: hypoglycemia avoidance (applicable = every at-risk patient-day) ---
glucose_events = con.execute(f"""
    SELECT cem.encounter_block, l.lab_collect_dttm, l.lab_value_numeric AS glucose
    FROM cohort_encounter_map cem
    JOIN '{labs_path}' l ON l.hospitalization_id = cem.hospitalization_id
    WHERE l.lab_category ILIKE '%glucose%' AND l.lab_value_numeric IS NOT NULL
""").df()
con.register("glucose_events", glucose_events)

# "Delivered" = avoided = no glucose < threshold recorded that day. Missing glucose data that day
# defaults to "avoided" (absence of a recorded low value, not proof none occurred) -- consistent
# with every other indicator here treating missing data as not-evaluable rather than non-compliant,
# though this one specifically can't distinguish "no event" from "not measured."
hypoglycemia = con.execute(f"""
    WITH hypo_days AS (
        SELECT DISTINCT ar.encounter_block, ar.day
        FROM at_risk ar
        JOIN glucose_events g ON g.encounter_block = ar.encounter_block
            AND g.lab_collect_dttm >= ar.day_start_dttm
            AND g.lab_collect_dttm < ar.day_start_dttm + INTERVAL '24 hours'
            AND g.glucose < {HYPOGLYCEMIA_THRESHOLD_MGDL}
    )
    SELECT
        (SELECT COUNT(*) FROM at_risk) AS n_applicable,
        (SELECT COUNT(*) FROM at_risk) - (SELECT COUNT(*) FROM hypo_days) AS n_delivered
""").df()
print(f"Hypoglycemia avoidance -- at-risk patient-days: {int(hypoglycemia['n_applicable'].iloc[0]):,}, "
      f"avoided: {int(hypoglycemia['n_delivered'].iloc[0]):,}")

In [ ]:
# --- 7g: combine into one pooled summary table ---
# All four indicators use the same Day 1..28 structure, but proning's day population (IMV days of
# ARDS-eligible encounters) isn't identical to the other three's (at_risk patient-days, which also
# require provider coverage and ICU location) -- each indicator's own rate is still valid on its
# own terms, just not built from directly interchangeable denominators.
ebp_summary = pd.DataFrame([
    {"indicator": "proning", "n_applicable": int(proning["n_applicable"].iloc[0]), "n_delivered": int(proning["n_delivered"].iloc[0])},
    {"indicator": "lpv_first_24h", "n_applicable": int(lpv_first24h["n_applicable"].iloc[0]), "n_delivered": int(lpv_first24h["n_delivered"].iloc[0])},
    {"indicator": "lpv_subsequent_days", "n_applicable": int(lpv_subsequent["n_applicable"].iloc[0]), "n_delivered": int(lpv_subsequent["n_delivered"].iloc[0])},
    {"indicator": "hypoglycemia_avoidance", "n_applicable": int(hypoglycemia["n_applicable"].iloc[0]), "n_delivered": int(hypoglycemia["n_delivered"].iloc[0])},
])
ebp_summary["rate"] = ebp_summary["n_delivered"] / ebp_summary["n_applicable"]
ebp_summary["suppressed_lt10"] = ebp_summary["n_applicable"] < 10

# pure aggregate (2 counts + 1 rate per indicator, no patient_id, no dates) -- no PHI, box is fine.
ebp_summary.to_csv("output_to_box/ebp_indicators.csv", index=False)
print(f"Wrote: output_to_box/ebp_indicators.csv")
ebp_summary